In [2]:
import pandas as pd
import numpy as np

In [3]:
# 0) 예시 데이터
data = pd.DataFrame({
    'trip_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'age': [25, 34, 45, 29, 40, 31, np.nan, 38, 27, 52],   # 제거할 컬럼 + 결측치 포함
    'car_color': ['Red', 'Blue', 'Green', 'Red', 'Black', 'White', 'Silver', 'Blue', 'Red', 'Black'],  # 제거할 컬럼
    'destination_from': ['Seoul', 'Busan', 'Incheon', 'Daegu', 'Daejeon', 'Seoul', 'Busan', 'Incheon', 'Daegu', 'Seoul'],
    'destination_to': ['Incheon', 'Ulsan', 'Seoul', 'Busan', 'Gwangju', 'Suwon', 'Daegu', 'Daejeon', 'Busan', 'Jeju'],
    'distance_km': [12.5, 25.3, 19.8, 30.2, 15.7, 8.4, 21.6, 17.9, 240.5, 13.2]  # 240.5는 이상치 예시
})

In [5]:
# 1) 불필요한 컬럼(age, car_color) 제거
data_cleaned = data.drop(columns = ['age','car_color'])

In [6]:
# 2) 최대 이동 거리 탐색 및 해당 경로 필터링
max_distance = data_cleaned['distance_km'].max()
longest_trip = data_cleaned[data_cleaned['distance_km'] == max_distance]

print("[결과] 가장 먼 거리의 출발지-도착지 경로")
print(longest_trip[['destination_from', 'destination_to', 'distance_km']])
print()

[결과] 가장 먼 거리의 출발지-도착지 경로
  destination_from destination_to  distance_km
8            Daegu          Busan        240.5



In [7]:
# 3) (전처리) 파생 변수 생성: 장거리 여부 플래그 (Feature Creation)
# 장거리 여부 표시
#    - 기준선을 하나 정해 두면(예: 20km) 집계·비율 확인이 쉬워짐
# is_long: 기준선 이상이면 1, 아니면 0 (type: int)
THRESHOLD_KM = 20  # 목적 기준선
data_feat = data_cleaned.copy()
data_feat['is_long'] = np.where(data_feat['distance_km'] >= THRESHOLD_KM,1,0)

print("[파생변수] is_long (>=20km):")
print(data_feat[['destination_from', 'destination_to', 'distance_km', 'is_long']])
print()

[파생변수] is_long (>=20km):
  destination_from destination_to  distance_km  is_long
0            Seoul        Incheon         12.5        0
1            Busan          Ulsan         25.3        1
2          Incheon          Seoul         19.8        0
3            Daegu          Busan         30.2        1
4          Daejeon        Gwangju         15.7        0
5            Seoul          Suwon          8.4        0
6            Busan          Daegu         21.6        1
7          Incheon        Daejeon         17.9        0
8            Daegu          Busan        240.5        1
9            Seoul           Jeju         13.2        0



In [8]:
# 4) (전처리) 결측치/이상치 점검
#    - 결측치 요약, IQR 기반 이상값 후보 탐지
# 4-1) 결측치 점검 (컬럼 별 결측치 개수)
na_summary = data_feat.isnull().sum()
print("[결측치 점검]")
print(na_summary)
print()

# 4-2) 이상치 점검 (IQR 방법) - 'distance_km'에 대한 이상치만 점검
q1 = data_feat['distance_km'].quantile(0.25)
q3 = data_feat['distance_km'].quantile(0.75)
iqr = q3-q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = data_feat[ (data_feat['distance_km'] < low) | (data_feat['distance_km'] > high) ]

print("[이상치(IQR) 점검]")
print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, 범위=({low:.2f} ~ {high:.2f})")
print("이상치 후보:\n", outliers if not outliers.empty else "없음")
print()

[결측치 점검]
trip_id             0
destination_from    0
destination_to      0
distance_km         0
is_long             0
dtype: int64

[이상치(IQR) 점검]
Q1=13.82, Q3=24.38, IQR=10.55, 범위=(-2.00 ~ 40.20)
이상치 후보:
    trip_id destination_from destination_to  distance_km  is_long
8      109            Daegu          Busan        240.5        1



In [9]:
# 5) (전처리) 범주형 인코딩 (One-Hot Encoding)
#    - 문자형(출발지/도착지)을 0/1 숫자 형태 컬럼으로 변환 - get_dummies 활용
encoded = pd.get_dummies(
    data_feat,
    columns=['destination_from', 'destination_to'],
    drop_first=False,
    dtype='int8'
)

print("[원-핫 인코딩 결과 컬럼]")
print(list(encoded.columns))
print()

[원-핫 인코딩 결과 컬럼]
['trip_id', 'distance_km', 'is_long', 'destination_from_Busan', 'destination_from_Daegu', 'destination_from_Daejeon', 'destination_from_Incheon', 'destination_from_Seoul', 'destination_to_Busan', 'destination_to_Daegu', 'destination_to_Daejeon', 'destination_to_Gwangju', 'destination_to_Incheon', 'destination_to_Jeju', 'destination_to_Seoul', 'destination_to_Suwon', 'destination_to_Ulsan']



In [10]:
# 6) 최종 전처리 산출물
#    - 불필요 컬럼 제거 + 파생변수 + 인코딩까지 마친 DataFrame 확인
print("[최종 전처리 데이터(preview)]")
print(encoded.head())

[최종 전처리 데이터(preview)]
   trip_id  distance_km  is_long  destination_from_Busan  \
0      101         12.5        0                       0   
1      102         25.3        1                       1   
2      103         19.8        0                       0   
3      104         30.2        1                       0   
4      105         15.7        0                       0   

   destination_from_Daegu  destination_from_Daejeon  destination_from_Incheon  \
0                       0                         0                         0   
1                       0                         0                         0   
2                       0                         0                         1   
3                       1                         0                         0   
4                       0                         1                         0   

   destination_from_Seoul  destination_to_Busan  destination_to_Daegu  \
0                       1                     0          